In [3]:
from sklearn.model_selection import train_test_split
from datasets import load_dataset
import pandas as pd
from peft import LoraConfig
from peft import get_peft_model, PeftModel
from transformers import TrainingArguments

import bitsandbytes as bnb
#from peft import create_loraplus_optimizer#----> not compatible with peft version==0.20

from transformers import Trainer

from datasets import Dataset
from pprint import pprint

from huggingface_hub import whoami
import torch

from transformers import AutoConfig

In [3]:
import os
#print(os.listdir("/kaggle/input/datasets/vaibhavraj46/model-folder-rag-generator-ft/rag3_llama_lora/checkpoint-200"))
#print(os.listdir("/kaggle/input/models/vaibhavraj46/rag3-llama-lora-final/transformers/rag3-llama-3.1-8b-qlora-iteration-1/1/rag3_llama_lora_final"))

In [2]:
#!pip install -U transformers peft bitsandbytes
!pip install bitsandbytes
#!pip install bert_score
#!pip install rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 41.2 MB/s eta 0:00:00:00:0100:01


**authenticate secret raed/write HF tokens**

In [4]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

from huggingface_hub import login

login(token=HF_TOKEN)

print(whoami())

{'type': 'user', 'id': '6a5e70dfab67e71b761fb3f3', 'name': 'vab46', 'fullname': 'Vaibhav Raj', 'isPro': False, 'avatarUrl': '/avatars/d689d000ad2f3233cb26a6ca99618cf1.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'colab-read-write', 'role': 'fineGrained', 'createdAt': '2026-08-16T12:11:43.414Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write'], 'scoped': [{'entity': {'_id': '6a5e70dfab67e71b761fb3f3', 'type': 'user', 'name': 'vab46'}, 'permissions': ['repo.content.read', 'repo.access.read', 'repo.write', 'discussion.write']}]}}}}


**diagnostic test**

In [5]:
#!pip install -q -U transformers==5.16.1
#!pip install transformers==5.16.1
import transformers
print(transformers.__version__)

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)

config = AutoConfig.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct"
)

print("Model type:", config.model_type)
print("Config class:", type(config).__name__)

5.0.0
Transformers: 5.0.0
PyTorch: 2.10.0+cu128


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

Model type: llama
Config class: LlamaConfig


In [6]:
import inspect
from transformers import TrainingArguments

#print(inspect.signature(TrainingArguments.__init__))

In [7]:
import peft
import inspect

print("PEFT version:", peft.__version__)
print("PEFT path:", peft.__file__)

print(
    "create_loraplus_optimizer in peft:",
    hasattr(peft, "create_loraplus_optimizer")
)

PEFT version: 0.19.1
PEFT path: /usr/local/lib/python3.12/dist-packages/peft/__init__.py
create_loraplus_optimizer in peft: False


**Load and validate final FT dataset and train-test split**

In [8]:
#HF_DATASET_ID = "vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA_ft"
HF_DATASET_ID = "vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA-junk_handled_ft"

# Load final cleaned dataset
rag3_ds = load_dataset(HF_DATASET_ID)

# Use train split (assuming the uploaded dataset has the default train split)
df_rag3 = rag3_ds["train"].to_pandas()

print("Rows:", len(df_rag3))
print("\nColumns:")
print(df_rag3.columns.tolist())

print("\nReference-answer types:")
print(df_rag3["reference_answer"].apply(type).value_counts())

print("\nStatus:")
print(df_rag3["status"].value_counts())

print("\nUnique documents:")
print(df_rag3["document_id"].nunique())

print("\nRows per document — summary:")
print(df_rag3.groupby("document_id").size().describe())

print("\nSample:")
display(
    df_rag3[
        ["anchor_id", "nctId", "document_id",
         "anchor_type", "anchor", "positive", "reference_answer"]
    ].head(3)
)

README.md:   0%|          | 0.00/621 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.32M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7866 [00:00<?, ? examples/s]

Rows: 7866

Columns:
['Unnamed: 0', 'anchor_id', 'nctId', 'document_id', 'anchor_type', 'anchor', 'positive', 'reference_answer', 'status', 'error']

Reference-answer types:
reference_answer
<class 'str'>    7866
Name: count, dtype: int64

Status:
status
success    7866
Name: count, dtype: int64

Unique documents:
2000

Rows per document — summary:
count    2000.000000
mean        3.933000
std         0.254054
min         2.000000
25%         4.000000
50%         4.000000
75%         4.000000
max         4.000000
dtype: float64

Sample:


,anchor_id,nctId,document_id,anchor_type,anchor,positive,reference_answer
0,0,NCT07640360,NCT07640360_1,macro_question,What is the main goal of this clinical trial r...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,The main goal of this clinical trial is to eva...
1,1,NCT07640360,NCT07640360_1,patient_profile_question,Could I qualify for this trial if I had a rece...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,"Yes, you could qualify for this trial if you h..."
2,2,NCT07640360,NCT07640360_1,operational_question,Is there an age limit for patients to particip...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,"Yes, there is an age limit for patients to par..."


In [9]:
# Document-level train/test split and their examples

SEED = 42
TEST_SIZE = 0.10

documents = df_rag3["document_id"].unique()

train_docs, test_docs = train_test_split(
    documents, test_size=TEST_SIZE, random_state=SEED
)

train_df = df_rag3[df_rag3["document_id"].isin(train_docs)].reset_index(drop=True)
test_df  = df_rag3[df_rag3["document_id"].isin(test_docs)].reset_index(drop=True)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"Train docs: {len(train_docs)} | Test docs: {len(test_docs)}")
print(f"Document overlap: {len(set(train_docs) & set(test_docs))}")

Train: 7082 | Test: 784
Train docs: 1800 | Test docs: 200
Document overlap: 0


In [10]:
train_df = train_df[["positive", "anchor", "reference_answer"]].copy()
test_df  = test_df[["positive", "anchor", "reference_answer"]].copy()

train_df.columns = ["context", "question", "answer"]
test_df.columns  = ["context", "question", "answer"]

print('+'*80)
print(f"Train examples: {len(train_df)}")
print(f"Test examples:  {len(test_df)}")

display(train_df.head(2))

++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
Train examples: 7082
Test examples:  784


,context,question,answer
0,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,What is the main goal of this clinical trial r...,The main goal of this clinical trial is to eva...
1,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,Could I qualify for this trial if I had a rece...,"Yes, you could qualify for this trial if you h..."


In [11]:
print("Train index:", train_df.index[:5].tolist(), "...", train_df.index[-1])
print("Test index:", test_df.index[:5].tolist(), "...", test_df.index[-1])

Train index: [0, 1, 2, 3, 4] ... 7081
Test index: [0, 1, 2, 3, 4] ... 783


**Load Llama 3.1 8B Instruct in 4-bit(QLoRA 4-bit) and set its config(only in 1st epoch), pad tokenizer to eos**

In [13]:
#!pip -q install -U transformers peft bitsandbytes accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

epoch=2# 1 if starting from base model, 2 if starting from ft model

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_ID = "vab46/llama-3.1-8b-instruct-lora-clinical_iter2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4: FP16
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if(epoch==1):
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16
    )
    
    model.config.use_cache = False
    
    print("Model loaded:", MODEL_ID)
    print("Compute dtype: FP16")
    print("Device:", model.device)

elif(epoch ==2):
    base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
    )
    
    base_model.config.use_cache = False
    
    # Load the already-trained LoRA adapter
    model = PeftModel.from_pretrained(
        base_model,
        ADAPTER_ID,
        is_trainable=True
    )
    
    model.config.use_cache = False

    print("Base model:", MODEL_ID)
    print("Existing LoRA adapter:", ADAPTER_ID)
    model.print_trainable_parameters()

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['monteclora_config', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Base model: meta-llama/Llama-3.1-8B-Instruct
Existing LoRA adapter: vab46/llama-3.1-8b-instruct-lora-clinical_iter2
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


Below block only for epoch 1

In [13]:
# RAG3 — Block 5: LoRA configuration(FOR 1ST EPOCH ONLY)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
    init_lora_weights=True
)

print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules='all-linear', exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [14]:
from transformers import AutoTokenizer
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

#model.config.pad_token_id = tokenizer.pad_token_id

print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)

PAD token: <|eot_id|>
PAD token ID: 128009
EOS token: <|eot_id|>
EOS token ID: 128009


**Convert to Llama instruction/chat format(static:-system_prompt; dynamic:- anchor and context as per records)**

In [15]:
#iter =1
iter =2

In [16]:
if(iter==1):
    SYSTEM_PROMPT = """You are a clinical-trial question answering assistant.

    Answer the user's QUESTION using the provided CONTEXT.

    - Answer directly and naturally.
    - Base the answer on the CONTEXT and do not introduce unsupported trial-specific facts.
    - Synthesize relevant information rather than simply copying the context.
    - Include important numbers, thresholds, dates, age ranges, conditions, interventions, and eligibility criteria when relevant.
    - For eligibility questions, explain whether the information given satisfies the relevant criteria and mention other important criteria when relevant.
    - Do not include irrelevant trial information.
    - If the CONTEXT does not provide enough information, say so clearly.
    - Preserve trial-specific details accurately.
    - Answer concisely, factually, and sufficiently completely.
    """
else:
    SYSTEM_PROMPT = """You are a clinical-trial question answering assistant.

    Answer the user's QUESTION using the provided CONTEXT.

    - Answer directly and naturally.
    - Base the answer on the CONTEXT and do not introduce unsupported trial-specific facts.
    - Synthesize relevant information rather than simply copying the context.
    - Include important numbers, thresholds, dates, age ranges, conditions, interventions, and eligibility criteria when relevant.
    - For eligibility questions, explain whether the information given satisfies the relevant criteria and mention other important criteria when relevant.
    - Do not include irrelevant trial information.
    - If the CONTEXT does not provide enough information, say so clearly.
    - Preserve trial-specific details accurately.
    - Answer concisely, factually, and sufficiently completely.
    - Don't limit answers to single word(1.eligiblity/qualification questions with a 'True/False'; 2. age/temporal questions with just a 'number') or blank(in case no answer avalilable in a given context).Rather 
      briefly explain the answer in with supported criteria/reasons(from context).
    - For eligiblity/qualification questions, preserve ALL relevant eligibility/inclusion criteria supported by the CONTEXT. If distinct criteria/reasons (generally >3-4) are relevant and long string based continous 
      sentence affects context, distinguish between criteria and additional criteria if supported by context .State the additional criteria explicitly(json). Else stay with string based continous format in other cases.
    """

def to_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"CONTEXT:\n{row['context']}\n\nQUESTION:\n{row['question']}"},
        {"role": "assistant", "content": row["answer"]}
    ]

train_df["messages"] = train_df.apply(to_messages, axis=1)
test_df["messages"] = test_df.apply(to_messages, axis=1)

display(train_df["messages"].iloc[0])

[{'role': 'system',
  'content': "You are a clinical-trial question answering assistant.\n\n    Answer the user's QUESTION using the provided CONTEXT.\n\n    - Answer directly and naturally.\n    - Base the answer on the CONTEXT and do not introduce unsupported trial-specific facts.\n    - Synthesize relevant information rather than simply copying the context.\n    - Include important numbers, thresholds, dates, age ranges, conditions, interventions, and eligibility criteria when relevant.\n    - For eligibility questions, explain whether the information given satisfies the relevant criteria and mention other important criteria when relevant.\n    - Do not include irrelevant trial information.\n    - If the CONTEXT does not provide enough information, say so clearly.\n    - Preserve trial-specific details accurately.\n    - Answer concisely, factually, and sufficiently completely.\n    - Don't limit answers to single word(1.eligiblity/qualification questions with a 'True/False'; 2. age

In [17]:
# Apply Llama chat template(serializes above messages into the exact Llama training format)

def format_chat(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

train_df["text"] = train_df["messages"].apply(format_chat)
test_df["text"]  = test_df["messages"].apply(format_chat)

print(train_df["text"].iloc[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a clinical-trial question answering assistant.

    Answer the user's QUESTION using the provided CONTEXT.

    - Answer directly and naturally.
    - Base the answer on the CONTEXT and do not introduce unsupported trial-specific facts.
    - Synthesize relevant information rather than simply copying the context.
    - Include important numbers, thresholds, dates, age ranges, conditions, interventions, and eligibility criteria when relevant.
    - For eligibility questions, explain whether the information given satisfies the relevant criteria and mention other important criteria when relevant.
    - Do not include irrelevant trial information.
    - If the CONTEXT does not provide enough information, say so clearly.
    - Preserve trial-specific details accurately.
    - Answer concisely, factually, and sufficiently completely.
    - Don't limit answers to

**convert to dataset**

In [18]:
train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df,
    preserve_index=False
)

#print("Train dataset:", train_dataset[:2])
#print("Test dataset:", test_dataset[:2])
print("Train dataset length:", len(train_dataset))
print("Test dataset length:", len(test_dataset))
print("Train dataset keys:", train_dataset.column_names)
print("Test dataset keys:", test_dataset.column_names)

Train dataset length: 7082
Test dataset length: 784
Train dataset keys: ['context', 'question', 'answer', 'messages', 'text']
Test dataset keys: ['context', 'question', 'answer', 'messages', 'text']


**Tokennize and genrate input_ids, mask_ids and lables**

In [19]:
MAX_LENGTH = 4096

def tokenize_and_mask(example):
    messages = example["messages"]

    # Full conversation: system + user + assistant
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Prompt only: system + user, ending at the assistant generation point
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize the complete sequence
    full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False,
    )

    # Tokenize prompt separately to identify where assistant answer starts
    prompt = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False,
    )

    input_ids = full["input_ids"]
    attention_mask = full["attention_mask"]

    prompt_length = min(len(prompt["input_ids"]), len(input_ids))

    # Ignore system + user tokens in the loss.
    # Only assistant answer tokens contribute to SFT loss.
    labels = [-100] * prompt_length + input_ids[prompt_length:]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

'''
tokenized_train_dataset = train_dataset.map(
    tokenize_and_mask,
    remove_columns=train_dataset.column_names,
)

tokenized_test_dataset = test_dataset.map(
    tokenize_and_mask,
    remove_columns=test_dataset.column_names,
)

print("Train columns:", tokenized_train_dataset.column_names)
print("Test columns:", tokenized_test_dataset.column_names)
print('~'*80)
'''
#end of fn--------------------------------------------------------------

'\ntokenized_train_dataset = train_dataset.map(\n    tokenize_and_mask,\n    remove_columns=train_dataset.column_names,\n)\n\ntokenized_test_dataset = test_dataset.map(\n    tokenize_and_mask,\n    remove_columns=test_dataset.column_names,\n)\n\nprint("Train columns:", tokenized_train_dataset.column_names)\nprint("Test columns:", tokenized_test_dataset.column_names)\nprint(\'~\'*80)\n'

In [20]:
tokenized_train_dataset = train_dataset.map(
    tokenize_and_mask,
    remove_columns=train_dataset.column_names,
)

tokenized_test_dataset = test_dataset.map(
    tokenize_and_mask,
    remove_columns=test_dataset.column_names,
)

print("Tokenized train columns:", tokenized_train_dataset.column_names)
print("Tokenized test columns:", tokenized_test_dataset.column_names)
print("Train examples:", len(tokenized_train_dataset))
print("Test examples:", len(tokenized_test_dataset))

Map:   0%|          | 0/7082 [00:00<?, ? examples/s]

Map:   0%|          | 0/784 [00:00<?, ? examples/s]

Tokenized train columns: ['input_ids', 'attention_mask', 'labels']
Tokenized test columns: ['input_ids', 'attention_mask', 'labels']
Train examples: 7082
Test examples: 784


**check samples(input_id, mask_id, labels):-all lengths equal+reproduciblity of Llama chat-template structure+ before the first answer token: labels = -100
from the answer onward: labels = input_ids**

In [21]:
sample = tokenized_train_dataset[0]

assert set(sample.keys()) == {"input_ids", "attention_mask", "labels"}
assert -100 not in sample["input_ids"]#-100 not in input_ids
assert len(sample["input_ids"]) == len(sample["attention_mask"]) == len(sample["labels"])#length same of all 3 fields
assert -100 in sample["labels"]#-100 in input_ids

first_answer = next(
    i for i, x in enumerate(sample["labels"]) if x != -100
)

#before the first answer token: labels = -100 from the answer onward: labels = input_ids
assert all(x == -100 for x in sample["labels"][:first_answer])
assert sample["labels"][first_answer] == sample["input_ids"][first_answer]

decoded = tokenizer.decode(
    sample["input_ids"],
    skip_special_tokens=False
)

print("Input length:", len(sample["input_ids"]))
print("First answer token position:", first_answer)
print("Non-masked labels:", sum(x != -100 for x in sample["labels"]))
print("Decoded input:\n", decoded)

Input length: 744
First answer token position: 686
Non-masked labels: 58
Decoded input:
 <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a clinical-trial question answering assistant.

    Answer the user's QUESTION using the provided CONTEXT.

    - Answer directly and naturally.
    - Base the answer on the CONTEXT and do not introduce unsupported trial-specific facts.
    - Synthesize relevant information rather than simply copying the context.
    - Include important numbers, thresholds, dates, age ranges, conditions, interventions, and eligibility criteria when relevant.
    - For eligibility questions, explain whether the information given satisfies the relevant criteria and mention other important criteria when relevant.
    - Do not include irrelevant trial information.
    - If the CONTEXT does not provide enough information, say so clearly.
    - Preserve trial-specific details accurately.
   

In [22]:
#tokenized_train_dataset[0]
# Free temporary generation memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

**Construct datacollatoir**:-We are not using DataCollatorForLanguageModeling, because that would construct its own labels and interfere with our assistant-only masking.

In [23]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

**Apply LoRA adapter (to the 4-bit model)-->ONLY IN 1ST EPOCH with Training configuration(with additional optimizer setup) finally followed by trainer constructor and then training**

In [24]:
#model = get_peft_model(model, lora_config)#iter 1
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [25]:
from transformers import TrainingArguments
import math

# Exact training-step calculation
effective_batch_size = 4 * 4
steps_per_epoch = math.ceil(len(train_df) / effective_batch_size)
warmup_steps = max(1, round(0.05 * steps_per_epoch))

training_args = TrainingArguments(
    #output_dir="./rag3_llama_lora",#for colab
    output_dir="/kaggle/working/rag3_llama_lora",# for kaggle notebook

    # ---- Training ----
    num_train_epochs=1,

    # ---- T4 VRAM ----
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    # ---- Optimization ----
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,# compatible with transformers==5.16, for higher version warmup_ratio also usable
    weight_decay=0.01,

    # ---- T4 precision ----
    fp16=True,
    bf16=False,

    # ---- Memory efficiency ----
    gradient_checkpointing=True,

    # ---- Logging ----
    logging_strategy="steps",
    logging_steps=50,
    logging_first_step=True,

    # ---- Checkpointing ----
    save_strategy="steps",
    save_steps=100,

    #save_strategy="epoch",# to avert issue of midway crash
    save_total_limit=1,

    # ---- Evaluation handled separately ----
    eval_strategy="no",

    # ---- Reproducibility ----
    seed=42,
    report_to="none",

    # ---- Data loading ----
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    remove_unused_columns=False,
)

print(f"Training examples: {len(train_df)}")
print(f"Effective batch size: {effective_batch_size}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Warmup steps: {warmup_steps}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Weight decay: {training_args.weight_decay}")

Training examples: 7082
Effective batch size: 16
Steps per epoch: 443
Warmup steps: 22
Learning rate: 0.0002
Weight decay: 0.01


In [27]:
optimizer = bnb.optim.Adam8bit(
    model.parameters(),
    lr=2e-4,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01,
)

print("Optimizer: bitsandbytes Adam8bit")
print("Learning rate:", 2e-4)
print("Betas:", (0.9, 0.999))
print("Weight decay:", 0.01)

Optimizer: bitsandbytes Adam8bit
Learning rate: 0.0002
Betas: (0.9, 0.999)
Weight decay: 0.01


In [28]:
# Free temporary generation memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    processing_class=tokenizer,
    optimizers=(optimizer, None),#One note: None for the scheduler lets Transformers create the scheduler from your training_args (cosine + warmup_steps).
    #This keeps the scheduler configuration centralized in TrainingArguments.
    data_collator=data_collator
)

print("Trainer ready.")

Trainer ready.


**One CPU/GPU batch sanity check before training**

In [30]:
batch = data_collator([tokenized_train_dataset[1]])

print("Batch shapes:")
print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:", batch["labels"].shape)#for initial start(no midway crash), refer below code


#after starting from ckpt again(post midway crash, or new iter, refer below code
#trainer.train(resume_from_checkpoint="/kaggle/input/datasets/vaibhavraj46/model-folder-rag-generator-ft/rag3_llama_lora/checkpoint-200")

print("Input contains -100:", (batch["input_ids"] == -100).any().item())
print("Labels contain -100:", (batch["labels"] == -100).any().item())

Batch shapes:
input_ids: torch.Size([1, 785])
attention_mask: torch.Size([1, 785])
labels: torch.Size([1, 785])
Input contains -100: False
Labels contain -100: True


**final_train**

In [31]:
#for initial start(no midway crash), refer below code
train_result = trainer.train()

print("Training loss:", train_result.training_loss)

#after starting from ckpt again(post midway crash, or new iter, refer below code
#trainer.train(resume_from_checkpoint="/kaggle/input/datasets/vaibhavraj46/model-folder-rag-generator-ft/rag3_llama_lora/checkpoint-200")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss
1,0.251259
50,0.183503
100,0.174509
150,0.163108
200,0.184803
250,0.179753
300,0.202597
350,0.223185
400,0.229568


Training loss: 0.19749066206725402


**save_temporary_on disc(with zip) & HF**

In [32]:
FINAL_ADAPTER_DIR = "./rag3_llama_lora_final"

trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

print(f"LoRA adapter saved to: {FINAL_ADAPTER_DIR}")



LoRA adapter saved to: ./rag3_llama_lora_final


In [34]:
import os
!zip -r /kaggle/working/rag3_llama_lora_final.zip \
    /kaggle/working/rag3_llama_lora_final

zip_path = "/kaggle/working/rag3_llama_lora_final.zip"

print("Exists:", os.path.exists(zip_path))
print("Size (MB):", os.path.getsize(zip_path) / (1024**2))

updating: kaggle/working/rag3_llama_lora_final/ (stored 0%)
updating: kaggle/working/rag3_llama_lora_final/adapter_config.json (deflated 59%)
updating: kaggle/working/rag3_llama_lora_final/tokenizer_config.json (deflated 43%)
updating: kaggle/working/rag3_llama_lora_final/chat_template.jinja (deflated 72%)
updating: kaggle/working/rag3_llama_lora_final/adapter_model.safetensors (deflated 7%)
updating: kaggle/working/rag3_llama_lora_final/tokenizer.json (deflated 85%)
updating: kaggle/working/rag3_llama_lora_final/training_args.bin (deflated 53%)
updating: kaggle/working/rag3_llama_lora_final/README.md (deflated 65%)
Exists: True
Size (MB): 150.60333347320557


In [35]:
print(train_df["messages"].iloc[0][1]['content'])
print(train_result.training_loss)
print(trainer.state.log_history)
#print(trainer.state(resume_from_checkpoint="/kaggle/working/kaggle/working/rag3_llama_lora_final").log_history)
print(trainer.args)

CONTEXT:
TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Guided Secondary Prevention After Stroke Randomized Trial of Nurse-led Program With Active vs Passive Smartwatch in Minor Stroke. A Randomized Controlled Trial Evaluating a Nurse-led Secondary Prevention and Physical Activity Program Supported by Either an Active Smartwatch (Structured Feedback) or Passive Smartwatch in Patients With Minor Stroke.
SUMMARY: After a first stroke or transient ischemic attack (TIA), the risk of recurrence is high in the weeks and months following the initial event. There are several modifiable risk factors that can reduce this risk, such as blood pressure, diet, physical activity, and smoking. Many stroke patients (NIHSS \< 5) have a low daily step count during the early recovery period, despite a good functional prognosis.
Active smartwatches provide real-time feedback, track progress, and set personalized walking goals, thereby boosting motivation and adherence to physical activity recommendations.
The

In [37]:
os.listdir('./rag3_llama_lora/checkpoint-443')

['optimizer.pt',
 'rng_state.pth',
 'adapter_config.json',
 'tokenizer_config.json',
 'chat_template.jinja',
 'adapter_model.safetensors',
 'tokenizer.json',
 'training_args.bin',
 'scheduler.pt',
 'README.md',
 'scaler.pt',
 'trainer_state.json']

In [42]:
import json
from huggingface_hub import ModelCard, CardData, HfApi

# ==========================================
# 1. SETUP PATHS
# ==========================================
repo_id = "vab46/llama-3.1-8b-instruct-lora-clinical_iter2_epoch2"#vab46/llama-3.1-8b-instruct-lora-clinical_iter2(for 2nd iter, 1st epoch)
#vab46/llama-3.1-8b-instruct-lora-clinical_iter1(for 1st epoch)
local_output_dir = "./rag3_llama_lora/checkpoint-443"

# Save the adapter weights, adapter_config.json, and trainer_state.json locally
trainer.save_model(local_output_dir) 

# ==========================================
# 2. EXTRACT LOGS & HYPERPARAMETERS
# ==========================================
args = trainer.args

try:
    with open(f"{local_output_dir}/trainer_state.json", "r") as f:
        state_data = json.load(f)
    log_history = state_data.get("log_history", [])

    # Generate the Markdown table for step losses
    loss_table = "| Step | Training Loss | Validation Loss |\n| :--- | :--- | :--- |\n"
    for log in log_history:
        step = log.get("step")
        t_loss = f"{log.get('loss'):.4f}" if "loss" in log else "-"
        v_loss = f"{log.get('eval_loss'):.4f}" if "eval_loss" in log else "-"
        if "loss" in log or "eval_loss" in log:
            loss_table += f"| {step} | {t_loss} | {v_loss} |\n"
except FileNotFoundError:
    loss_table = "*Training metrics log not found.*"

# ==========================================
# 3. BUILD THE STRUCTURED MODEL CARD
# ==========================================
card_data = CardData(
    base_model="meta-llama/Llama-3.1-8B-Instruct",
    license="llama3.1",
    language="en",
    tags=["lora", "peft", "transformers", "text-generation"],
    library_name="peft"
)

card_content = f"""
# Llama-3.1-8B-Instruct LoRA Fine-Tuned

This model is a PEFT/LoRA adapter single epoch fine-tuned on top of 
[meta-llama/Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct). 
Fine tuning has been done on 
[iter2 anchor-context-groundTruth_junkHandled dataset](https://huggingface.co/vab46/llama-3.1-8b-instruct-lora-clinical_iter2/tree/main/vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA-junk_handled_ft) 
related to clinical trials. This fine tuned adapter on base LLM can be to address clinical trials related questions in 
general(macro/broad overview, patient eligiblity, operations etc related question). 
Further this can also be used for prompt engineering based on CT, or RAG generation(related to CT) 
based on retrieved chunks.

# Model Details:-

## Model Description:-

- **Model Type**: Causal Large Language Model (Autoregressive Decoder),
- **Base model**: meta-llama/Llama-3.1-8B-Instruct,
- **Maximum Sequence Length**: 131,072 tokens (128K context window),
- **Output Dimensionality (Hidden Size)**: 4096 dimensions,
- **Supported Modality**: Text,
- **Training Dataset**: [iter2 anchor-context-groundTruth_junkHandled dataset](vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA-junk_handled_ft) (mapped via Chat Templates),
- **Language**: en,
- **License**: llama3.1 (Subject to Meta's Llama 3.1 Community License Agreement)

## Model Sources:-
- **Documentation**:-Transformers Documentation / PEFT Documentation,
- **Repository**: Transformers on GitHub,
- **Hugging Face**: PEFT Adapters Hub

## How to Load and Run Inference

You can load this model using `bitsandbytes` for 4-bit quantization and `peft` to apply the adapter weights. Tokenizer formatting utilizes the base model's official chat template to map query/question tokens alongside retrieved context blocks.

```python
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# 1. Load the tokenizer from the base model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 2. Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 3. Load quantized base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

# 4. Load your current LoRA fine-tuned adapter
model = PeftModel.from_pretrained(
    base_model,
    "{repo_id}"
)
model.eval()

# ==========================================
# 5. STRUCTURE RAG PROMPTS WITH CHAT TEMPLATE
# ==========================================
# Mapping the query/question with relevant context chunks into the message schema
SYSTEM_PROMPT = "You are a helpful assistant specialized in clinical data analysis."
retrieved_context = "..."  # Paste or map your retrieved context blocks here
user_query = "..."         # Paste your question/anchor here

messages = [
    {{"role": "system", "content": SYSTEM_PROMPT}},
    {{
        "role": "user", 
        "content": f"CONTEXT(RETRIEVED_BLOCKS):\n{train_df["messages"].iloc[0][1]['content']}"
    }}
]

# Apply Llama 3.1 specific formatting tokens automatically
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# Generate response
with torch.no_grad():
    outputs = model.generate(
        inputs, 
        max_new_tokens=512, 
        eos_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print("RELEVANT_ANSWER(generated by LLM):", response)
```

## Training Hyperparameters
The following configurations were used during training:
- **Train Batch Size:** {getattr(args, 'per_device_train_batch_size', 'N/A')}
- **gradient_accumulation_steps:** {getattr(args, 'gradient_accumulation_steps', 'N/A')},
- **Total Epochs:** {getattr(args, 'num_train_epochs', 'N/A')}
- **Weight Decay:** {getattr(args, 'weight_decay', 'N/A')}
- **Learning Rate:** {getattr(args, 'learning_rate', 'N/A')}
- **Output Directory**: {getattr(args, 'output_dir', 'N/A')}
- **LR Scheduler Type:** {getattr(args, 'lr_scheduler_type', 'N/A')}
- **Warmup Steps:** {getattr(args, 'warmup_steps', 'N/A')}
- **FP16 Precision:** {getattr(args, 'fp16', 'N/A')}
- **BF16 Precision:** {getattr(args, 'bf16', 'N/A')}
- **Gradient Checkpointing:** {getattr(args, 'gradient_checkpointing', 'N/A')}
- **Logging Strategy:** {getattr(args, 'logging_strategy', 'N/A')}
- **Logging Steps:** {getattr(args, 'logging_steps', 'N/A')}
- **Logging First Step:** {getattr(args, 'logging_first_step', 'N/A')}
- **Save Strategy:** {getattr(args, 'save_strategy', 'N/A')}
- **Save Steps:** {getattr(args, 'save_steps', 'N/A')}
- **#**save_strategy:**, "epoch",#use this (ignore previous 2 stepwise checkpoint) if strong gpu +VRAM, else use stepwise ckpt
- **Save Total Limit:** {getattr(args, 'save_total_limit', 'N/A')}
- **Evaluation Strategy:** {getattr(args, 'eval_strategy', 'N/A')}
- **Seed:** {getattr(args, 'seed', 'N/A')}
- **Report To:** {getattr(args, 'report_to', 'N/A')}
- **Dataloader Num Workers:** {getattr(args, 'dataloader_num_workers', 'N/A')}
- **Dataloader Pin Memory:** {getattr(args, 'dataloader_pin_memory', 'N/A')}
- **Remove Unused Columns:** {getattr(args, 'remove_unused_columns', 'N/A')}

## Training Loss History
{loss_table}
"""

# Save the complete README.md locally into the weight folder
card = ModelCard.from_template(card_data, template_str=card_content)
card.save(f"{local_output_dir}/README.md")

# ==========================================
# 4. EXPLICIT UPLOAD VIA HF API (Bypassing Trainer)
# ==========================================
api = HfApi()

# Creates the repo on HF if it does not already exist
api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")


# Uploads the entire folder including weights and the programmatic README.md
api.upload_folder(
    folder_path=local_output_dir,
    repo_id=repo_id,
    repo_type="model"
)
print(f"Successfully uploaded adapter configuration and model card to {repo_id}")


#trainer.model.push_to_hub("vab46/rag3-llama-lora-2")# suffix 1 for 1st iter, 2 for 2nd iter

Repo card metadata block was not found. Setting CardData to empty.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Successfully uploaded adapter configuration and model card to vab46/llama-3.1-8b-instruct-lora-clinical_iter2_epoch2


## **Evaluation**

In [43]:
!pip install bert_score
!pip install rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.6 MB/s eta 0:00:00


In [44]:
import torch
import pandas as pd
from tqdm.auto import tqdm
import os

import sys
import subprocess
import importlib.util

import re
import pandas as pd
import numpy as np

from rouge_score import rouge_scorer
from bert_score import score as bertscore

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel


### **Loading model:-**

**common_protion**

In [45]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

**via 1)zip uploaded**

In [ ]:
#!unzip -q /kaggle/input/rag3_llama_lora_final.zip -d /kaggle/working/

In [ ]:
ft1_model = PeftModel.from_pretrained(
    base_model,
    #"/kaggle/working/rag3_llama_lora_final"
    "/kaggle/input/models/vaibhavraj46/rag3-llama-lora-final/transformers/rag3-llama-3.1-8b-qlora-iteration-1/1/rag3_llama_lora_final",
    #automatically unzipped by kaggle when uploaded
)

ft1_model.eval()

**and 2)HF**

In [47]:
repo_id

'vab46/llama-3.1-8b-instruct-lora-clinical_iter2_epoch2'

In [48]:
ft1_model = PeftModel.from_pretrained(
    base_model,
    repo_id
)

ft1_model.eval()

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

**Generate Base vs LoRA predictions**

In [49]:
# Evaluation configuration — keep fixed
# --------------------------------------------------
EVAL_MAX_NEW_TOKENS = 256
EVAL_TEMPERATURE = 0.3#for candidate_model
EVAL_TOP_P = 0.9

#model.eval()

def build_eval_prompt(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"CONTEXT:\n{row['context']}\n\n"
                f"QUESTION:\n{row['question']}"
            )
        }
    ]


# --------------------------------------------------
# Batched generation for DataFrame rows
# --------------------------------------------------

EVAL_BATCH_SIZE = 8

# Decoder-only LLMs should use left padding for batched generation
tokenizer.padding_side = "left"


def generate_answers_batch(model, tokenizer, df_batch):
    prompts = []

    for _, row in df_batch.iterrows():
        messages = build_eval_prompt(row)

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=EVAL_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=EVAL_TEMPERATURE,
            top_p=EVAL_TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Because inputs are left-padded, all generated tokens begin
    # after the common padded input length.
    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

    predictions = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )

    return [prediction.strip() for prediction in predictions]


**diagnosis**

In [53]:
print(base_model.device)

cuda:0


fixing pad token issue and adding same to configs

In [51]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

base_model.config.pad_token_id = tokenizer.pad_token_id
ft1_model.config.pad_token_id = tokenizer.pad_token_id

### base pred

In [52]:
# --------------------------------------------------
# 1. Base Llama 3.1 8B Instruct
# --------------------------------------------------

base_predictions = []

for start in tqdm(
    range(0, len(test_df), EVAL_BATCH_SIZE),
    desc="Base Llama 3.1 8B"
):
    batch_df = test_df.iloc[
        start:start + EVAL_BATCH_SIZE
    ]

    batch_predictions = generate_answers_batch(
        base_model,
        tokenizer,
        batch_df
    )

    base_predictions.extend(batch_predictions)

print("Base predictions:", len(base_predictions))

Base Llama 3.1 8B:   0%|          | 0/98 [00:00<?, ?it/s]

Base predictions: 784


### LoRA pred

In [54]:
# --------------------------------------------------
# 2. LoRA-1 Llama 3.1 8B Instruct
# --------------------------------------------------

lora_predictions = []

for start in tqdm(
    range(0, len(test_df), EVAL_BATCH_SIZE),
    desc="LoRA-1 Llama 3.1 8B"
):
    batch_df = test_df.iloc[
        start:start + EVAL_BATCH_SIZE
    ]

    batch_predictions = generate_answers_batch(
        ft1_model,
        tokenizer,
        batch_df
    )

    lora_predictions.extend(batch_predictions)

print("LoRA predictions:", len(lora_predictions))

LoRA-1 Llama 3.1 8B:   0%|          | 0/98 [00:00<?, ?it/s]

LoRA predictions: 784


**df_generation**

In [55]:
assert len(base_predictions) == len(test_df)
assert len(lora_predictions) == len(test_df)

In [56]:
eval_df = test_df[
    ["context", "question", "answer"]
].copy()

eval_df["base_prediction"] = base_predictions
eval_df["lora_prediction"] = lora_predictions

print("\nEvaluation complete.")
print("Examples:", len(eval_df))
print("Base empty predictions:", eval_df["base_prediction"].eq("").sum())
print("LoRA empty predictions:", eval_df["lora_prediction"].eq("").sum())


Evaluation complete.
Examples: 784
Base empty predictions: 0
LoRA empty predictions: 0


**Save evaluation predictions**

In [57]:
EVAL_DIR = "/kaggle/working/rag3_evaluation"
os.makedirs(EVAL_DIR, exist_ok=True)

EVAL_PATH = os.path.join(EVAL_DIR, "base_vs_lora_predictions.csv")

eval_df.to_csv(EVAL_PATH, index=False)

print(f"Saved: {EVAL_PATH}")
print(f"Rows: {len(eval_df)}")
print(f"File size: {os.path.getsize(EVAL_PATH) / (1024**2):.2f} MB")

Saved: /kaggle/working/rag3_evaluation/base_vs_lora_predictions.csv
Rows: 784
File size: 1.76 MB


**Basic prediction sanity checks**

In [58]:
# Basic prediction sanity checks
# --------------------------------------------------

for col in ["base_prediction", "lora_prediction"]:
    empty = eval_df[col].fillna("").astype(str).str.strip().eq("").sum()
    nulls = eval_df[col].isna().sum()
    lengths = eval_df[col].fillna("").astype(str).str.split().str.len()

    print(f"\n{col}")
    print("-" * len(col))
    print("Null predictions :", nulls)
    print("Empty predictions:", empty)
    print("Min words        :", lengths.min())
    print("Median words     :", lengths.median())
    print("Max words        :", lengths.max())

# Verify expected evaluation size
assert len(eval_df) == 784#783 for iter1, 784 for iter2

# Verify no missing predictions
assert eval_df["base_prediction"].notna().all()
assert eval_df["lora_prediction"].notna().all()

print("\n✓ Basic prediction checks passed.")


base_prediction
---------------
Null predictions : 0
Empty predictions: 0
Min words        : 3
Median words     : 33.0
Max words        : 128

lora_prediction
---------------
Null predictions : 0
Empty predictions: 0
Min words        : 3
Median words     : 33.0
Max words        : 116

✓ Basic prediction checks passed.


**Robust automated scoring setup + scoring**:-

This block computes:

- ROUGE-L
- BERTScore
- answer/reference semantic similarity
- basic groundedness/unsupported-content diagnostics using the supplied context

It also saves the scored dataframe immediately.

In [59]:
# BLOCK 4 — Automated reference + semantic scoring
# ============================================================

# Install only if missing
if importlib.util.find_spec("rouge_score") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rouge-score"])

if importlib.util.find_spec("bert_score") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bert-score"])


# 1. ROUGE-L(comparsion of base candidate answer with ref_answer)
# ------------------------------------------------------------

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

def rouge_l_f1(reference, prediction):
    return rouge.score(
        str(reference),
        str(prediction)
    )["rougeL"].fmeasure

eval_df["base_rougeL"] = [
    rouge_l_f1(r, p)
    for r, p in zip(eval_df["answer"], eval_df["base_prediction"])
]

eval_df["lora_rougeL"] = [
    rouge_l_f1(r, p)
    for r, p in zip(eval_df["answer"], eval_df["lora_prediction"])
]

# 2. BERTScore(comparsion of base candidate answer with ref_answer)
# ------------------------------------------------------------

references = eval_df["answer"].astype(str).tolist()
base_preds = eval_df["base_prediction"].astype(str).tolist()
lora_preds = eval_df["lora_prediction"].astype(str).tolist()

print("Computing BERTScore: Base...")
_, _, base_f1 = bertscore(
    base_preds,
    references,
    lang="en",
    verbose=True,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Computing BERTScore: LoRA...")
_, _, lora_f1 = bertscore(
    lora_preds,
    references,
    lang="en",
    verbose=True,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

eval_df["base_bertscore_f1"] = base_f1.cpu().numpy()
eval_df["lora_bertscore_f1"] = lora_f1.cpu().numpy()

# 3. Save immediately
# ------------------------------------------------------------

AUTO_SCORE_PATH = os.path.join(
    EVAL_DIR,
    "base_vs_lora_automated_scores.csv"
)

eval_df.to_csv(AUTO_SCORE_PATH, index=False)

print("\n✓ Automated scoring complete.")
print("Saved:", AUTO_SCORE_PATH)

print("\nMean scores:")
print(
    eval_df[
        [
            "base_rougeL",
            "lora_rougeL",
            "base_bertscore_f1",
            "lora_bertscore_f1"
        ]
    ].mean()
)

Computing BERTScore: Base...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/23 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/13 [00:00<?, ?it/s]

done in 15.19 seconds, 51.61 sentences/sec

✓ Automated scoring complete.
Saved: /kaggle/working/rag3_evaluation/base_vs_lora_automated_scores.csv

Mean scores:
base_rougeL          0.681817
lora_rougeL          0.683128
base_bertscore_f1    0.943086
lora_bertscore_f1    0.943949
dtype: float64


**Clinical-detail / groundedness checks**:- Interpretation: numeric recall is a completeness diagnostic, while numeric groundedness is a hallucination-risk diagnostic(roughly>90% although it does not pretend to be a perfect hallucination detector).

In [60]:
# BLOCK 5 — Clinical-detail / groundedness diagnostics
# ============================================================
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def extract_numbers(text):
    """
    Extract numeric expressions including:
    integers, decimals, percentages, ranges, and common units.
    """
    text = str(text).lower()

    patterns = [
        r"\b\d+(?:\.\d+)?\s*%",
        r"\b\d+(?:\.\d+)?\s*(?:mg|g|kg|mcg|ml|l|cm|mm|years?|months?|weeks?|days?|hours?|minutes?)\b",
        r"\b\d+(?:\.\d+)?\s*[-–]\s*\d+(?:\.\d+)?\b",
        r"\b\d+(?:\.\d+)?\b",
    ]

    values = []
    for pattern in patterns:
        values.extend(re.findall(pattern, text))

    return set(values)

def numeric_support_ratio(context, prediction):
    """
    Fraction of numeric expressions in the prediction
    that are also supported by the supplied context.
    """
    pred_nums = extract_numbers(prediction)

    if not pred_nums:
        return 1.0

    context_norm = normalize_text(context)

    supported = sum(
        normalize_text(n) in context_norm
        for n in pred_nums
    )

    return supported / len(pred_nums)

def numeric_reference_recall(reference, prediction):
    """
    Fraction of numeric expressions in the reference
    that are reproduced in the prediction.
    Diagnostic only: absence is not automatically an error.
    """
    ref_nums = extract_numbers(reference)

    if not ref_nums:
        return 1.0

    pred_norm = normalize_text(prediction)

    matched = sum(
        normalize_text(n) in pred_norm
        for n in ref_nums
    )

    return matched / len(ref_nums)


# Compute diagnostics
# ------------------------------------------------------------

eval_df["base_numeric_groundedness"] = [
    numeric_support_ratio(c, p)
    for c, p in zip(
        eval_df["context"],
        eval_df["base_prediction"]
    )
]

eval_df["lora_numeric_groundedness"] = [
    numeric_support_ratio(c, p)
    for c, p in zip(
        eval_df["context"],
        eval_df["lora_prediction"]
    )
]

eval_df["base_numeric_recall"] = [
    numeric_reference_recall(r, p)
    for r, p in zip(
        eval_df["answer"],
        eval_df["base_prediction"]
    )
]

eval_df["lora_numeric_recall"] = [
    numeric_reference_recall(r, p)
    for r, p in zip(
        eval_df["answer"],
        eval_df["lora_prediction"]
    )
]

# Flag potentially unsupported numeric details
# ------------------------------------------------------------

eval_df["base_numeric_issue"] = (
    eval_df["base_numeric_groundedness"] < 1.0
)

eval_df["lora_numeric_issue"] = (
    eval_df["lora_numeric_groundedness"] < 1.0
)

print("Base numeric-grounding issues:",
      eval_df["base_numeric_issue"].sum())

print("LoRA numeric-grounding issues:",
      eval_df["lora_numeric_issue"].sum())

print("\nMean diagnostics:")
print(
    eval_df[
        [
            "base_numeric_groundedness",
            "lora_numeric_groundedness",
            "base_numeric_recall",
            "lora_numeric_recall"
        ]
    ].mean()
)

# Save updated evaluation
# ------------------------------------------------------------

eval_df.to_csv(
    os.path.join(
        EVAL_DIR,
        "base_vs_lora_scored.csv"
    ),
    index=False
)

print("\n✓ Clinical-detail diagnostics complete.")

Base numeric-grounding issues: 64
LoRA numeric-grounding issues: 66

Mean diagnostics:
base_numeric_groundedness    0.968105
lora_numeric_groundedness    0.965843
base_numeric_recall          0.900501
lora_numeric_recall          0.903364
dtype: float64

✓ Clinical-detail diagnostics complete.


**Aggregate Base vs LoRA + task-type stratification**:- It uses the scores we already computed and does not regenerate anything.

In [61]:
# BLOCK 6 — Aggregate comparison + task stratification
# ============================================================

import numpy as np
import pandas as pd

# 1. Overall Base vs LoRA comparison
# ------------------------------------------------------------

metric_pairs = {
    "ROUGE-L": ("base_rougeL", "lora_rougeL"),
    "BERTScore-F1": ("base_bertscore_f1", "lora_bertscore_f1"),
    "Numeric groundedness": (
        "base_numeric_groundedness",
        "lora_numeric_groundedness"
    ),
    "Numeric recall": (
        "base_numeric_recall",
        "lora_numeric_recall"
    ),
}

overall_rows = []

for metric, (base_col, lora_col) in metric_pairs.items():
    base_mean = eval_df[base_col].mean()
    lora_mean = eval_df[lora_col].mean()

    overall_rows.append({
        "Metric": metric,
        "Base": base_mean,
        "LoRA-1": lora_mean,
        "Absolute Δ": lora_mean - base_mean,
        "Relative Δ %": (
            (lora_mean - base_mean) / base_mean * 100
            if base_mean != 0 else np.nan
        )
    })

overall_results = pd.DataFrame(overall_rows)

print("OVERALL RESULTS")
print("=" * 75)
print(overall_results.to_string(index=False))

# 2. Define task categories from the questions
# ------------------------------------------------------------

def classify_question(question):
    q = str(question).lower()

    if any(x in q for x in [
        "eligible", "eligibility", "qualify",
        "qualification", "inclusion", "exclusion"
    ]):
        return "Eligibility / Qualification"

    if any(x in q for x in [
        "how many", "how much", "percentage", "%",
        "dose", "mg", "years", "months", "weeks",
        "days", "threshold", "duration"
    ]):
        return "Numeric / Threshold"

    if any(x in q for x in [
        "treatment", "intervention", "drug",
        "therapy", "dose", "administer"
    ]):
        return "Intervention"

    if any(x in q for x in [
        "when", "date", "before", "after",
        "during", "duration", "period"
    ]):
        return "Temporal"

    if any(x in q for x in [
        "what type", "study design", "randomized",
        "randomised", "placebo", "blind",
        "phase", "trial"
    ]):
        return "Study Design"

    return "Other"

eval_df["question_type"] = (
    eval_df["question"]
    .map(classify_question)
)

print("\nQUESTION-TYPE DISTRIBUTION")
print("=" * 45)
print(eval_df["question_type"].value_counts())

# 3. Stratified automated performance
# ------------------------------------------------------------

stratified = (
    eval_df
    .groupby("question_type")
    .agg(
        examples=("question", "size"),

        base_rougeL=("base_rougeL", "mean"),
        lora_rougeL=("lora_rougeL", "mean"),

        base_bertscore=("base_bertscore_f1", "mean"),
        lora_bertscore=("lora_bertscore_f1", "mean"),

        base_num_ground=("base_numeric_groundedness", "mean"),
        lora_num_ground=("lora_numeric_groundedness", "mean"),

        base_num_recall=("base_numeric_recall", "mean"),
        lora_num_recall=("lora_numeric_recall", "mean"),
    )
    .reset_index()
)

print("\nSTRATIFIED RESULTS")
print("=" * 100)
print(stratified.to_string(index=False))

# 4. Save
# ------------------------------------------------------------

overall_results.to_csv(
    os.path.join(EVAL_DIR, "overall_results.csv"),
    index=False
)

stratified.to_csv(
    os.path.join(EVAL_DIR, "stratified_results.csv"),
    index=False
)

eval_df.to_csv(
    os.path.join(EVAL_DIR, "final_scored_predictions.csv"),
    index=False
)

print("\n✓ Aggregate and stratified evaluation complete.")

OVERALL RESULTS
              Metric     Base   LoRA-1  Absolute Δ  Relative Δ %
             ROUGE-L 0.681817 0.683128    0.001311      0.192302
        BERTScore-F1 0.943086 0.943949    0.000863      0.091516
Numeric groundedness 0.968105 0.965843   -0.002261     -0.233601
      Numeric recall 0.900501 0.903364    0.002864      0.318008

QUESTION-TYPE DISTRIBUTION
question_type
Eligibility / Qualification    228
Study Design                   227
Other                          180
Intervention                    64
Temporal                        48
Numeric / Threshold             37
Name: count, dtype: int64

STRATIFIED RESULTS
              question_type  examples  base_rougeL  lora_rougeL  base_bertscore  lora_bertscore  base_num_ground  lora_num_ground  base_num_recall  lora_num_recall
Eligibility / Qualification       228     0.635210     0.655508        0.927499        0.932038         0.919022         0.907919         0.851973         0.871438
               Intervention      

**General context/unsupport-support diagnostic(left in block4) + paired comparison(b/w base and LORA ft in context of Rouge & BERT score)**

In [62]:
# BLOCK 7 — General context support + paired Base vs LoRA
# ============================================================

import re
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer

rouge_diag = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

# 1. Sentence-level context-support diagnostic
# ------------------------------------------------------------

def split_sentences(text):
    text = str(text).strip()
    if not text:
        return []
    return [
        s.strip()
        for s in re.split(r"(?<=[.!?])\s+", text)
        if s.strip()
    ]

def context_support_score(context, prediction):
    """
    For each prediction sentence, find its maximum
    lexical support (ROUGE-L F1) against a context sentence.
    
    Diagnostic proxy only:
    low support does NOT automatically mean hallucination,
    because valid paraphrases may have low lexical overlap.
    """
    context_sentences = split_sentences(context)
    prediction_sentences = split_sentences(prediction)

    if not prediction_sentences or not context_sentences:
        return np.nan

    scores = []

    for pred_sent in prediction_sentences:
        best = max(
            rouge_diag.score(ctx_sent, pred_sent)["rougeL"].fmeasure
            for ctx_sent in context_sentences
        )
        scores.append(best)

    return float(np.mean(scores))


eval_df["base_context_support"] = [
    context_support_score(c, p)
    for c, p in zip(
        eval_df["context"],
        eval_df["base_prediction"]
    )
]

eval_df["lora_context_support"] = [
    context_support_score(c, p)
    for c, p in zip(
        eval_df["context"],
        eval_df["lora_prediction"]
    )
]

# 2. Unsupported-content proxy
# ------------------------------------------------------------

# We deliberately call this a "low-support sentence" diagnostic,
# NOT a hallucination rate.

SUPPORT_THRESHOLD = 0.10

def low_support_sentence_rate(context, prediction):
    context_sentences = split_sentences(context)
    prediction_sentences = split_sentences(prediction)

    if not prediction_sentences or not context_sentences:
        return 0.0

    low_support = 0

    for pred_sent in prediction_sentences:
        best = max(
            rouge_diag.score(ctx_sent, pred_sent)["rougeL"].fmeasure
            for ctx_sent in context_sentences
        )
        if best < SUPPORT_THRESHOLD:
            low_support += 1

    return low_support / len(prediction_sentences)


eval_df["base_low_support_rate"] = [
    low_support_sentence_rate(c, p)
    for c, p in zip(
        eval_df["context"],
        eval_df["base_prediction"]
    )
]

eval_df["lora_low_support_rate"] = [
    low_support_sentence_rate(c, p)
    for c, p in zip(
        eval_df["context"],
        eval_df["lora_prediction"]
    )
]

# 3. Paired Base vs LoRA comparison
# ------------------------------------------------------------

eval_df["rouge_winner"] = np.where(
    eval_df["lora_rougeL"] > eval_df["base_rougeL"],
    "LoRA",
    np.where(
        eval_df["lora_rougeL"] < eval_df["base_rougeL"],
        "Base",
        "Tie"
    )
)

eval_df["bertscore_winner"] = np.where(
    eval_df["lora_bertscore_f1"] > eval_df["base_bertscore_f1"],
    "LoRA",
    np.where(
        eval_df["lora_bertscore_f1"] < eval_df["base_bertscore_f1"],
        "Base",
        "Tie"
    )
)

print("PAIRED ROUGE-L OUTCOMES")
print(eval_df["rouge_winner"].value_counts())

print("\nPAIRED BERTSCORE OUTCOMES")
print(eval_df["bertscore_winner"].value_counts())

# 4. Overall context-support comparison
# ------------------------------------------------------------

support_summary = pd.DataFrame({
    "Metric": [
        "Mean context-support score",
        "Mean low-support sentence rate",
        "Mean numeric groundedness",
        "Mean numeric recall"
    ],
    "Base": [
        eval_df["base_context_support"].mean(),
        eval_df["base_low_support_rate"].mean(),
        eval_df["base_numeric_groundedness"].mean(),
        eval_df["base_numeric_recall"].mean()
    ],
    "LoRA-1": [
        eval_df["lora_context_support"].mean(),
        eval_df["lora_low_support_rate"].mean(),
        eval_df["lora_numeric_groundedness"].mean(),
        eval_df["lora_numeric_recall"].mean()
    ]
})

print("\nGROUNDEDNESS / COMPLETENESS DIAGNOSTICS")
print("=" * 70)
print(support_summary.to_string(index=False))

# 5. Save complete evaluation table
# ------------------------------------------------------------

FINAL_EVAL_PATH = os.path.join(
    EVAL_DIR,
    "rag3_complete_evaluation.csv"
)

eval_df.to_csv(
    FINAL_EVAL_PATH,
    index=False
)

support_summary.to_csv(
    os.path.join(
        EVAL_DIR,
        "groundedness_summary.csv"
    ),
    index=False
)

print("\n✓ Final evaluation diagnostics saved.")
print("File:", FINAL_EVAL_PATH)

PAIRED ROUGE-L OUTCOMES
rouge_winner
Tie     365
LoRA    211
Base    208
Name: count, dtype: int64

PAIRED BERTSCORE OUTCOMES
bertscore_winner
LoRA    275
Base    268
Tie     241
Name: count, dtype: int64

GROUNDEDNESS / COMPLETENESS DIAGNOSTICS
                        Metric     Base   LoRA-1
    Mean context-support score 0.442570 0.441146
Mean low-support sentence rate 0.022534 0.022534
     Mean numeric groundedness 0.968105 0.965843
           Mean numeric recall 0.900501 0.903364

✓ Final evaluation diagnostics saved.
File: /kaggle/working/rag3_evaluation/rag3_complete_evaluation.csv


In [63]:
!zip -q -r rag3_evaluation.zip /kaggle/working/rag3_evaluation

## save the final model

In [ ]:
'''
FINAL_ADAPTER_DIR = "./rag3_llama_lora_final"

trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

print(f"LoRA adapter saved to: {FINAL_ADAPTER_DIR}")
'''